In [ ]:
!pip install langgraph

In [1]:
import os
from typing import TypedDict, Annotated, Optional
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# langchain
# chain = prompt | retriever | llm | parser
# node : 함수 (search, find, ...)
# edge : node 들을 연결하는 연결선
# state : 그래프 전체에서 공유하는 dictionary

In [2]:
from langchain_core.runnables import RunnableLambda

In [3]:
chain = RunnableLambda(lambda x : x.upper()) | RunnableLambda(lambda x : x + "!")
print(chain.invoke('hello'))

HELLO!


In [8]:
class MiniState(TypedDict):
    text : str
    
def upper_node(state : MiniState) -> dict:
    return {"text": state['text'].upper()}

def excite_node(state):
    return {"text" : state['text'] + '!'}


b = StateGraph(MiniState)
b.add_node('upper', upper_node)
b.add_node('excite', excite_node)
b.add_edge(START, 'upper')
b.add_edge('upper', 'excite')
b.add_edge('excite', END)
app = b.compile()
app.invoke({'text' : 'hello'})

{'text': 'HELLO!'}

In [9]:
class SimpleState(TypedDict):
    message : str
        
def greet(state: SimpleState) -> dict:
    return {"message" : f"안녕, {state['message']}"}

builder= StateGraph(SimpleState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)
app = builder.compile()
app.invoke({'message' : '월요일'})

{'message': '안녕, 월요일'}

In [10]:
# name 키를 받아서 "안녕하세요, {name}님!" 을 리턴
class WelcomeState(TypedDict):
    name : str
    greeting : str

def welcome(state: WelcomeState) -> dict:
    return {greeting' : f"환영합니다, {state['name']}님!"}

builder = StateGraph(WelcomeState)
builder.add_node('welcome', welcome)
builder.add_edge(START, 'welcome')
builder.add_edge('welcome', END)
app = builder.compile()
app.invoke({'name' : '모두연', 'greeting' : ""})

{'name': '모두연', 'greeting': '환영합니다, 모두연님!'}

In [11]:
class ChatState(TypedDict):
    user_msg : str
    reply : str
    turn : int

class AnalaysisState(TypedDict):
    text : str
    word_count : int
    sentiment : str

b1 = StateGraph(ChatState)
b2 = StateGraph(AnalaysisState)
type(b1).__name__, type(b2).__name__

('StateGraph', 'StateGraph')

In [12]:
class UserState(TypedDict):
    user_id : str
    name : str
    age : int
    is_active : bool
        
b1 = StateGraph(UserState)
UserState.__annotations__

{'user_id': str, 'name': str, 'age': int, 'is_active': bool}

In [13]:
class CounterState(TypedDict):
    count : int
    note : str
    
def add_one(state : CounterState) -> dict:   # state : {'count' : 1, 'note' : 'abc'}
    return {'count' : state['count']+1}

b = StateGraph(CounterState)
b.add_node('add_one', add_one)
b.add_edge(START, 'add_one')
b.add_edge('add_one', END)
b.compile().invoke({'count' : 0, 'note': '안녕하세요'})

{'count': 1, 'note': '안녕하세요'}

In [14]:
class ScoreState(TypedDict):
    name : str
    score : int
    bonus : int

def bump_score(state:ScoreState) -> dict:
    return {'score' : state['score']+ 10}

# def bump_score(state:ScoreState) -> dict:
#     state['score'] +=10
#     return state  # {'name' : 원래이름, 'score' : 원래스코어 + 10 , 'bonus' : 원래 bonus}

In [ ]:
# multi-node serial

In [18]:
class TextState(TypedDict):
    text : str
    upper : str
    length : int

def to_upper(state: TextState) -> dict:
    return {'upper' : state['text'].upper()}

def measure(state: TextState) -> dict:
    return {'length' : len(state['text'])}

builder = StateGraph(TextState)
builder.add_node('to_upper', to_upper)
builder.add_node('measure', measure)

builder.add_edge(START, 'to_upper')
builder.add_edge('to_upper', 'measure')
builder.add_edge('measure', END)

app = builder.compile()
app.invoke({'text' : 'hello langgraph', 'upper' : '', 'length' : 0})

{'text': 'hello langgraph', 'upper': 'HELLO LANGGRAPH', 'length': 15}

In [22]:
class DocState(TypedDict): 
    raw : str          
    cleaned : str      #: strip()
    translated : str   #: upper()
    summary : str      #: [:5]
    log : str

def clean(state:DocState) -> dict:
    return {'cleaned' : state['raw'].strip(), 'log' : state['log'] + 'clean,'}

def translate(state:DocState) -> dict:
    return {'translated' : state['cleaned'].upper(), 'log' : state['log'] + 'translate,'}

def summarize(state:DocState) -> dict:
    return {'summary' : state['translated'][:5], 'log' : state['log'] + 'summarize,'}

builder = StateGraph(DocState)
for n, fn in [('clean', clean), ('translate', translate), ('summarize', summarize)]:
    builder.add_node(n, fn)
    
builder.add_edge(START, 'clean')
builder.add_edge('clean', 'translate')
builder.add_edge('translate', 'summarize')
builder.add_edge('summarize', END)
app = builder.compile()

In [23]:
app.invoke({"raw": "  hello langgraph world  ", "cleaned": "", "translated": "", "summary": "", 'log' : ''})

{'raw': '  hello langgraph world  ',
 'cleaned': 'hello langgraph world',
 'translated': 'HELLO LANGGRAPH WORLD',
 'summary': 'HELLO',
 'log': 'clean,translate,summarize,'}

In [24]:
class QAState(TypedDict):
    question : str
    answer : str
        
def llm_answer(state : QAState) -> dict:
    response = llm.invoke([HumanMessage(content = state['question'])])
    return {'answer' : response.content}

builder = StateGraph(QAState)
builder.add_node('llm_answer', llm_answer)
builder.add_edge(START, 'llm_answer')
builder.add_edge('llm_answer', END)
app = builder.compile()
app.invoke({'question' : 'langgraph를 한 문장으로?', 'answer' : ''})

{'question': 'langgraph를 한 문장으로?',
 'answer': 'LangGraph는 다양한 언어를 이해하고 처리하는 데 도움을 주는 인공지능 기반의 언어 모델입니다.'}

In [27]:
class TransState(TypedDict):
    ko : str
    en : str

def translate(state : TransState) -> dict:
    response = llm.invoke([
        SystemMessage(content = 'translate to english, only english'),
        HumanMessage(content = state['ko'])])
    return {'en' : response.content}

builder = StateGraph(TransState)
builder.add_node('translate', translate)
builder.add_edge(START, 'translate')
builder.add_edge('translate', END)
app = builder.compile()
app.invoke({"ko": "오늘 날씨가 좋네요.", "en": ""})

{'ko': '오늘 날씨가 좋네요.', 'en': 'The weather is nice today.'}

In [ ]:
# builder.add_conditional_edge

In [31]:
class NumberState(TypedDict):
    n : int
    label : str
        
def start_n(state):
    return {}

def even_handler(state):
    return {'label' : f"{state['n']}은(는) 짝수입니다"}


def odd_handler(state):
    return {'label' : f"{state['n']}은(는) 홀수입니다"}

def parity_router(state) -> str:
    return "even" if state['n']%2 ==0 else 'odd'

b = StateGraph(NumberState)
b.add_node('start_n', start_n)
b.add_node('even_handler', even_handler)
b.add_node('odd_handler', odd_handler)

b.add_edge(START, 'start_n')
b.add_conditional_edges('start_n', parity_router, {'even' : 'even_handler', 'odd':'odd_handler'})
b.add_edge('even_handler' , END)
b.add_edge('odd_handler', END)
app = b.compile()

In [32]:
app.invoke({'n' : 7, 'label' : ''})

{'n': 7, 'label': '7은(는) 홀수입니다'}

In [33]:
app.invoke({'n' : 8, 'label' : ''})

{'n': 8, 'label': '8은(는) 짝수입니다'}

In [36]:
class ExamState(TypedDict) : 
    score : int
    result : str
        
def start_exam(state):
    return {}
def pass_node(state) : return {'result' : '합격입니다'}
def fail_node(state) : return {'result' : '불합격입니다'}

def exam_router(state) -> str:
    return 'pass' if state['score'] >=60 else 'fail'


b = StateGraph(ExamState)
b.add_node('start_exam', start_exam)
b.add_node('pass_node', pass_node)
b.add_node('fail_node', fail_node)

b.add_edge(START, 'start_exam')
b.add_conditional_edges('start_exam', exam_router, {'pass' : 'pass_node', 'fail':'fail_node'})
b.add_edge('pass_node' , END)
b.add_edge('fail_node', END)
app = b.compile()    

In [37]:
print(app.invoke({"score": 85, "result": ""}))
print(app.invoke({"score": 40, "result": ""}))

{'score': 85, 'result': '합격입니다'}
{'score': 40, 'result': '불합격입니다'}


In [38]:
class GradeState(TypedDict):
    score : int
    grade : str

def start_grade(state): return {}

def grade_a(state) : return {'grade': 'A'}
def grade_b(state) : return {'grade': 'B'}
def grade_c(state) : return {'grade': 'C'}

def grade_router(state) -> str:
    if state['score'] >=90 : return 'a'
    if state['score'] >= 70: return 'b'
    return 'c'

b = StateGraph(GradeState)
b.add_node('start_grade', start_grade)
b.add_node('grade_a', grade_a)
b.add_node('grade_b', grade_b)
b.add_node('grade_c', grade_c)

b.add_edge(START, 'start_grade')
b.add_conditional_edges('start_grade', grade_router, 
                       {'a' : 'grade_a', 'b' : 'grade_b', 'c' : 'grade_c'})

for n in ['grade_a', 'grade_b', 'grade_c']:
    b.add_edge(n, END)
app = b.compile()

In [39]:
for s in [95, 75, 50]:
    print(app.invoke({'score':s, 'grade': ''}))

{'score': 95, 'grade': 'A'}
{'score': 75, 'grade': 'B'}
{'score': 50, 'grade': 'C'}


In [43]:
class QuestionState(TypedDict):
    question : str
    qtype : str
    answer : str

def h_when(state):
    return {'answer' : '[시간] 답변'}
def h_where(state):
    return {'answer' : '[장소] 답변'}
def h_what(state):
    return {'answer' : '[일반] 답변'}
def h_stmt(state):
    return {'answer' : '[진술문] 검색 불필요'}

def merge(state):
    return {'answer' : state['answer'] + '!!!!!!!'}

def classify(state):
    q = state['question'].lower()
    if '?' not in q: return {'qtype' : 'statement'}
    if any(w in q for w in ['언제', 'when']): return {'qtype' : 'when'}
    if any(w in q for w in ['어디', 'where']): return {'qtype' : 'where'}
    return {'qtype' : 'what'}

In [45]:
b = StateGraph(QuestionState)
b.add_node('classify', classify)
# b.add_node('h_when', h_when)
for name, fn in [('h_when', h_when), ('h_where', h_where), ('h_what', h_what), ('h_stmt', h_stmt)]:
    b.add_node(name, fn)
b.add_node('merge', merge)

b.add_edge(START, 'classify')
b.add_conditional_edges('classify', lambda s: s['qtype'], 
                       {'when' :'h_when', 'where' : 'h_where', 'what' : 'h_what', 'statement' : 'h_stmt' })

b.add_edge('h_when', 'merge')
b.add_edge('h_where', 'merge')
b.add_edge('h_what', 'merge')
b.add_edge('h_stmt', 'merge')
b.add_edge('merge', END)
app = b.compile()

In [46]:
app.invoke({'question' : "회의 어디서 해?", 'qtype' : '', 'answer' : ''})

{'question': '회의 어디서 해?', 'qtype': 'where', 'answer': '[장소] 답변!!!!!!!'}